In [ ]:
# imports
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
import sys


PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))


SHORT_NAMES = {
    # unique models
    "lda_evaluation_results": "LDA",
    "linear_svm_evaluation_results": "Linear SVM",
    "xgboost_model_evaluation": "XGBoost",
    "ripper_model_evaluation": "RIPPER",
    "threshold_tuned_autoencoder+MLP_evaluation_results": "AE+MLP (tuned)",

    # baseline ensembles
    "Baseline_model_with_entropy_aggregation_results": "Entropy Aggregation",
    "baseline_trimmed_mean_results": "Trimmed Mean",
    "Baseline_model_with_max_probability_aggregation_results": "Max Prob Aggregation",
    "Baseline_model_with_threshold_aware_aggregation_results": "Threshold-Aware Aggregation",
    "Baseline_model_with_weighted_average_aggregation_results": "Weighted Avg Aggregation",
}


In [39]:
def _read_json(path: str) -> dict:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

In [40]:
def _positive_class_key(classification_report: dict) -> str:
    if "1" in classification_report:
        return "1"

    ignore = {"accuracy", "macro avg", "weighted avg"}
    class_keys = [k for k in classification_report.keys() if k not in ignore]

    supports = []
    for k in class_keys:
        sup = classification_report.get(k, {}).get("support", None)
        if sup is not None:
            supports.append((k, float(sup)))

    if supports:
        supports.sort(key=lambda x: x[1])  # smallest support first
        return supports[0][0]

    return class_keys[0] if class_keys else "1"

In [41]:
# function that loads all the json files (results)

def load_results(json_paths):
    """
    Always uses filename stem as the key, so your lists are stable.
    """
    results = {}
    for path in json_paths:
        raw = _read_json(path)
        stem = os.path.splitext(os.path.basename(path))[0]
        results[stem] = raw
    return results

In [42]:
# function to build a clean table of metrics for plotting

def _metrics_table(results_dict, model_names):
    rows = []
    for name in model_names:
        raw = results_dict[name]
        cr = raw.get("classification_report", {})
        pos = _positive_class_key(cr)

        rows.append({
            "model": name,
            "precision": float(cr[pos]["precision"]),
            "recall": float(cr[pos]["recall"]),
            "f1": float(cr[pos]["f1-score"]),
            "roc_auc": float(raw.get("roc_auc", np.nan)),
        })

    df = pd.DataFrame(rows).set_index("model")
    return df

In [43]:
def _ensure_dir(out_dir):
    os.makedirs(out_dir, exist_ok=True)


def _save_png(fig, out_path):
    fig.savefig(out_path, dpi=200, bbox_inches="tight", format="png")
    plt.close(fig)

In [44]:
# Data visuaisations

def _bar_plot(df, column, title, out_path):
    fig = plt.figure(figsize=(max(8, len(df) * 1.2), 5))
    ax = fig.add_subplot(111)
    ax.bar(df.index, df[column].values)
    ax.set_title(title)
    ax.set_ylim(0, 1.05)
    ax.set_ylabel(column)
    ax.set_xticklabels(df.index, rotation=25, ha="right")
    fig.tight_layout()
    _save_png(fig, out_path)

In [45]:
def _plot_roc_curves(results_dict, model_names, title, out_path, label_map=None):
    fig = plt.figure(figsize=(7, 6))
    ax = fig.add_subplot(111)

    plotted = False
    for name in model_names:
        raw = results_dict[name]
        rc = raw.get("roc_curve", None)
        if not rc:
            continue
        fpr = np.array(rc.get("fpr", []), dtype=float)
        tpr = np.array(rc.get("tpr", []), dtype=float)
        if len(fpr) == 0 or len(tpr) == 0:
            continue

        auc = raw.get("roc_auc", None)
        display = label_map.get(name, name) if label_map else name
        label = f"{display} (AUC={auc:.3f})" if isinstance(auc, (int, float)) else display

        ax.plot(fpr, tpr, label=label)
        plotted = True

    ax.plot([0, 1], [0, 1], linestyle="--", label="Chance")
    ax.set_title(title)
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.legend()

    fig.tight_layout()
    if plotted:
        _save_png(fig, out_path)
    else:
        plt.close(fig)
        print(f"[INFO] No ROC curve arrays found for: {title}")


In [46]:
# function to compare unique models (precision, recall, roc, f1)

def compare_unique_models(results_dict, unique_model_names, out_dir="comparison_png"):
    _ensure_dir(out_dir)

    df = _metrics_table(results_dict, unique_model_names)
    df = df.rename(index=SHORT_NAMES)


    _bar_plot(df, "precision", "Unique Models: Precision (positive class)", os.path.join(out_dir, "unique_precision.png"))
    _bar_plot(df, "recall",    "Unique Models: Recall (positive class)",    os.path.join(out_dir, "unique_recall.png"))
    _bar_plot(df, "f1",        "Unique Models: F1 (positive class)",        os.path.join(out_dir, "unique_f1.png"))
    _bar_plot(df, "roc_auc",   "Unique Models: ROC-AUC",                    os.path.join(out_dir, "unique_roc_auc.png"))

    _plot_roc_curves(results_dict, unique_model_names, "Unique Models: ROC Curves",
                 os.path.join(out_dir, "unique_roc_curves.png"),
                 label_map=SHORT_NAMES)


    return df

In [47]:
# function to compare baseline models (precision, recall, roc, f1)

def compare_baseline_models(results_dict, baseline_model_names, out_dir="comparison_png"):
    _ensure_dir(out_dir)

    df = _metrics_table(results_dict, baseline_model_names)
    df = df.rename(index=SHORT_NAMES)


    _bar_plot(df, "precision", "Baseline Ensembles: Precision (positive class)", os.path.join(out_dir, "baseline_precision.png"))
    _bar_plot(df, "recall",    "Baseline Ensembles: Recall (positive class)",    os.path.join(out_dir, "baseline_recall.png"))
    _bar_plot(df, "f1",        "Baseline Ensembles: F1 (positive class)",        os.path.join(out_dir, "baseline_f1.png"))
    _bar_plot(df, "roc_auc",   "Baseline Ensembles: ROC-AUC",                    os.path.join(out_dir, "baseline_roc_auc.png"))

    _plot_roc_curves(results_dict, baseline_model_names, "Baseline Ensembles: ROC Curves",
                 os.path.join(out_dir, "baseline_roc_curves.png"),
                 label_map=SHORT_NAMES)


    return df

In [48]:
# function to compare 5 unique and the best baseline model (precision, recall, roc, f1)

def compare_unique_vs_best_baseline(results_dict, unique_model_names, baseline_model_names,
                                   out_dir="comparison_png", best_by="f1"):

    _ensure_dir(out_dir)

    df_unique = _metrics_table(results_dict, unique_model_names)
    df_base   = _metrics_table(results_dict, baseline_model_names)

    best_baseline_name = df_base[best_by].idxmax()
    df_combined = pd.concat([df_unique, df_base.loc[[best_baseline_name]]], axis=0)

    df_plot = df_combined.rename(index=SHORT_NAMES)



    # Bar plots
    _bar_plot(df_plot, "precision",
              f"Unique Models + Best Baseline: Precision (best baseline = {best_baseline_name})",
              os.path.join(out_dir, "unique_vs_bestbaseline_precision.png"))

    _bar_plot(df_plot, "recall",
              f"Unique Models + Best Baseline: Recall (best baseline = {best_baseline_name})",
              os.path.join(out_dir, "unique_vs_bestbaseline_recall.png"))

    _bar_plot(df_plot, "f1",
              f"Unique Models + Best Baseline: F1 (best baseline = {best_baseline_name})",
              os.path.join(out_dir, "unique_vs_bestbaseline_f1.png"))

    _bar_plot(df_plot, "roc_auc",
              f"Unique Models + Best Baseline: ROC-AUC (best baseline = {best_baseline_name})",
              os.path.join(out_dir, "unique_vs_bestbaseline_roc_auc.png"))

    # Recall-Precision scatter plot
    fig = plt.figure(figsize=(7, 6))
    ax = fig.add_subplot(111)
    ax.scatter(df_plot["recall"], df_plot["precision"])

    for model_name in df_plot.index:
        ax.annotate(model_name,
                    (df_plot.loc[model_name, "recall"], df_plot.loc[model_name, "precision"]),
                    fontsize=9, xytext=(4, 4), textcoords="offset points")

    ax.set_title(f"Unique vs Best Baseline (best_by={best_by}: {best_baseline_name})")
    ax.set_xlabel("Recall (positive class)")
    ax.set_ylabel("Precision (positive class)")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    fig.tight_layout()
    _save_png(fig, os.path.join(out_dir, "unique_vs_bestbaseline_recall_precision.png"))


     # ROC curves for unique models vs best baseline
    subset_names = list(df_unique.index) + [best_baseline_name]

    _plot_roc_curves(
        results_dict,
        subset_names,
        title=f"Unique + Best Baseline: ROC Curves (best baseline = {SHORT_NAMES.get(best_baseline_name, best_baseline_name)})",
        out_path=os.path.join(out_dir, "unique_vs_bestbaseline_roc_curves.png"),
        label_map=SHORT_NAMES
    )

    return best_baseline_name, df_combined


In [ ]:
if __name__ == "__main__":
    json_paths = [
        PROJECT_ROOT / "Advance Models/results/lda_evaluation_results.json",
        PROJECT_ROOT / "Advance Models/results/linear_svm_evaluation_results.json",
        PROJECT_ROOT / "Advance Models/results/xgboost_model_evaluation.json",
        PROJECT_ROOT / "Advance Models/results/ripper_model_evaluation.json",
        PROJECT_ROOT / "Advance Models/results/threshold_tuned_autoencoder+MLP_evaluation_results.json",
        PROJECT_ROOT / "Baseline Models/results/Baseline_model_with_entropy_aggregation_results.json",
        PROJECT_ROOT / "Baseline Models/results/baseline_trimmed_mean_results.json",
        PROJECT_ROOT / "Baseline Models/results/Baseline_model_with_max_probability_aggregation_results.json",
        PROJECT_ROOT / "Baseline Models/results/Baseline_model_with_threshold_aware_aggregation_results.json",
        PROJECT_ROOT / "Baseline Models/results/Baseline_model_with_weighted_average_aggregation_results.json",
    ]

    results = load_results(json_paths)

    unique_models = [
        "lda_evaluation_results",
        "threshold_tuned_autoencoder+MLP_evaluation_results",
        "linear_svm_evaluation_results",
        "ripper_model_evaluation",
        "xgboost_model_evaluation",
    ]

    baseline_models = [
        "Baseline_model_with_entropy_aggregation_results",
        "baseline_trimmed_mean_results",
        "Baseline_model_with_max_probability_aggregation_results",
        "Baseline_model_with_threshold_aware_aggregation_results",
        "Baseline_model_with_weighted_average_aggregation_results",
    ]


    out_dir = PROJECT_ROOT /"Comparative Evaluation of Advanced and Baseline Models"

    compare_unique_models(results, unique_models, out_dir=out_dir)
    compare_baseline_models(results, baseline_models, out_dir=out_dir)
    best_name, combined_df = compare_unique_vs_best_baseline(
        results, unique_models, baseline_models, out_dir=out_dir, best_by="f1"
    )

/tmp/ipython-input-177599119.py:10: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(df.index, rotation=25, ha="right")
/tmp/ipython-input-177599119.py:10: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(df.index, rotation=25, ha="right")
/tmp/ipython-input-177599119.py:10: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(df.index, rotation=25, ha="right")
/tmp/ipython-input-177599119.py:10: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(df.index, rotation=25, ha="right")
/tmp/ipython-input-177599119.py:10: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e.

In [50]:
print(f"All PNGs saved to: {out_dir}")
print(f"Best baseline picked by F1: {best_name}")

All PNGs saved to: comparison_png
Best baseline picked by F1: baseline_trimmed_mean_results
